# 21 — Confronto sul test ufficiale e con il benchmark (T37)

**Zero training**: rileggiamo le predizioni dei notebook15–20, tutte sul medesimo
KDDTest+. Verifichiamo input, protocollo, parametri, etichette e conteggi prima di
confrontare le metriche. Ogni modello è stato addestrato sul 100% di KDDTrain+.

Il paper di Divekar, TabellaIV, riporta F1 per classe e weighted F1 su NSL-KDD
senza SMOTE. Non confrontiamo il nostro macro F1 con il loro weighted F1.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_recall_fscore_support, f1_score, accuracy_score

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.data import COLUMN_NAMES
from src.targets import AUTHOR_ATTACK_TO_FAMILY, FAMILY_ORDER
OUTPUT = ROOT / 'artifacts/official_benchmark_comparison'
OUTPUT.mkdir(parents=True, exist_ok=True)
protocol_path = ROOT / 'configs/official_test_v1.json'
protocol = json.loads(protocol_path.read_text(encoding='utf-8'))
protocol_hash = hashlib.sha256(protocol_path.read_bytes()).hexdigest()
for name, digest in protocol['input_sha256'].items():
    assert hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == digest
test = pd.read_csv(ROOT / 'data/raw/KDDTest+.csv', header=None, names=COLUMN_NAMES)
assert test['attack_label'].isin(AUTHOR_ATTACK_TO_FAMILY).all()
expected_truth = test['attack_label'].map(AUTHOR_ATTACK_TO_FAMILY).to_numpy(dtype=str)
models = {
    'decision_tree': ('Decision Tree', 'decision_tree_official_test'),
    'random_forest_balanced': ('Random Forest', 'random_forest_balanced_official_test'),
    'gaussian_naive_bayes': ('Gaussian Naive Bayes', 'gaussian_naive_bayes_official_test'),
    'svm_rbf': ('SVM RBF', 'svm_rbf_official_test'),
    'neural_network': ('Rete neurale', 'neural_network_official_test'),
    'kmeans': ('K-Means', 'kmeans_official_test'),
}

## 1. Caricare, verificare e ricalcolare

Predizioni con target e indici diversi, parametri non conformi o conteggi JSON
incoerenti fermano il confronto. Nessun risultato di validation interna entra
in questa classifica.

In [ ]:
overall_rows = []
class_rows = []
matrices = {}
saved_results = {}
for model_id, (label, artifact) in models.items():
    folder = ROOT / 'artifacts' / artifact
    result = json.loads((folder / 'results.json').read_text(encoding='utf-8'))
    assert result['protocol_sha256'] == protocol_hash
    assert result['protocol_id'] == protocol['protocol_id']
    assert result['input_sha256'] == protocol['input_sha256']
    assert result['label_profile'] == protocol['label_profile']
    assert result['family_order'] == list(FAMILY_ORDER)
    assert result['train_rows'] == 125973 and result['test_rows'] == len(expected_truth)
    assert result['classifier_parameters'] == protocol['models'][model_id]['parameters']
    assert hashlib.sha256((folder / 'predictions.npz').read_bytes()).hexdigest() == result['predictions_sha256']
    with np.load(folder / 'predictions.npz', allow_pickle=False) as saved:
        truth, predicted = saved['truth'], saved['predicted']
        np.testing.assert_array_equal(saved['test_indices'], np.arange(len(expected_truth)))
    np.testing.assert_array_equal(truth, expected_truth)
    assert set(predicted) <= set(FAMILY_ORDER)
    counts = confusion_matrix(truth, predicted, labels=FAMILY_ORDER)
    normalized = confusion_matrix(truth, predicted, labels=FAMILY_ORDER, normalize='true')
    np.testing.assert_array_equal(counts, result['confusion_counts'])
    np.testing.assert_allclose(normalized, result['confusion_normalized'])
    precision, recall, f1, support = precision_recall_fscore_support(truth, predicted, labels=FAMILY_ORDER, zero_division=0)
    scores = {
        'macro_f1': f1_score(truth, predicted, labels=FAMILY_ORDER, average='macro', zero_division=0),
        'weighted_f1': f1_score(truth, predicted, labels=FAMILY_ORDER, average='weighted', zero_division=0),
        'accuracy': accuracy_score(truth, predicted),
    }
    for metric, value in scores.items():
        np.testing.assert_allclose(value, result['metrics'][metric])
    overall_rows.append({'model_id': model_id, 'model': label, **scores, 'fit_seconds': result['fit_seconds']})
    for i, family in enumerate(FAMILY_ORDER):
        class_rows.append({'model_id': model_id, 'model': label, 'family': family,
                           'precision': precision[i], 'recall': recall[i], 'f1': f1[i],
                           'support': int(support[i]), 'true_positives': int(counts[i,i]),
                           'false_positives': int(counts[:,i].sum()-counts[i,i]),
                           'error_rate': 1-recall[i]})
    matrices[model_id] = normalized
    saved_results[model_id] = result
overall = pd.DataFrame(overall_rows)
per_class = pd.DataFrame(class_rows)

## 2. Confronto tra i nostri sei modelli

Ordiniamo per macro F1, la metrica primaria stabilita nel protocollo. Affianchiamo
le metriche per classe; non eleggiamo un modello affidabile sulla sola accuracy.

In [ ]:
display(overall.sort_values('macro_f1', ascending=False).round(4))
display(per_class[per_class['family'].isin(['R2L','U2R'])].round(4))

In [ ]:
fig, axes = plt.subplots(2,3,figsize=(17,10))
for axis, (model_id, (label, _)) in zip(axes.flat, models.items()):
    ConfusionMatrixDisplay(matrices[model_id], display_labels=FAMILY_ORDER).plot(ax=axis, cmap='Blues', values_format='.2f', colorbar=False, im_kw={'vmin':0,'vmax':1})
    axis.set_title(label)
    axis.set_xlabel('Famiglia predetta')
    axis.set_ylabel('Famiglia vera')
fig.suptitle('Test ufficiale — confusion matrix normalizzate per classe vera')
fig.tight_layout()
fig.savefig(OUTPUT / 'six_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Confronto con Divekar — Tabella IV, NSL-KDD senza SMOTE

Fonte: [paper](https://arxiv.org/pdf/1811.05372), TabellaIV, PDF pagina6.
Valori del paper arrotondati a percentuali intere. Ordine qui ricondotto a
Normal, DoS, Probe, R2L, U2R. Delta in **punti percentuali**: nostro meno paper.
Confronto descrittivo, non prova di superiorità statistica o replica numerica.

In [ ]:
# Trascrizione della riga NSL-KDD per ogni modello, verificata sul paper locale.
# Colonne: Normal, DoS, Probe, R2L, U2R, weighted F1 (percentuali).
paper = pd.DataFrame.from_dict({
    'decision_tree': [77,84,66,3,12,68],
    'random_forest_balanced': [79,87,70,0,4,70],
    'gaussian_naive_bayes': [73,63,7,29,4,57],
    'svm_rbf': [80,84,72,31,12,74],
    'neural_network': [79,85,74,3,5,71],
    'kmeans': [78,84,52,3,3,68],
}, orient='index', columns=[*FAMILY_ORDER, 'weighted_f1'])
paper.index.name = 'model_id'
weighted_comparison = overall[['model_id','model','weighted_f1']].copy()
weighted_comparison['our_weighted_f1_pct'] = weighted_comparison.pop('weighted_f1')*100
weighted_comparison['paper_weighted_f1_pct'] = weighted_comparison['model_id'].map(paper['weighted_f1'])
weighted_comparison['delta_pp'] = weighted_comparison['our_weighted_f1_pct'] - weighted_comparison['paper_weighted_f1_pct']
class_comparison = per_class[['model_id','model','family','f1']].copy()
class_comparison['our_f1_pct'] = class_comparison.pop('f1')*100
class_comparison['paper_f1_pct'] = [paper.loc[row.model_id, row.family] for row in class_comparison.itertuples()]
class_comparison['delta_pp'] = class_comparison['our_f1_pct'] - class_comparison['paper_f1_pct']
display(weighted_comparison.round(2))
display(class_comparison.round(2))

## 4. Limiti del confronto

- Stesso tipo di task e test ufficiale, ma non identiche condizioni sperimentali.
- Noi: 41 feature, one-hot, configurazioni congelate; nessuna ricerca di parametri.
- Paper/codice autore: scelte di selezione, encoding e configurazioni differenti;
  K-Means del sorgente è MiniBatchKMeans, il nostro è Lloyd. Dettagli nell'audit T27.
- Righe dei nostri file 125973/22544 contro 125972/22542 dichiarate nel paper;
  nessuna eliminazione arbitraria per uguagliare i conteggi.
- Una configurazione per modello: niente media, deviazione o intervalli da questi
  sei valori. I tempi sono indicativi di questa macchina, non un benchmark prestazionale.
- Il test è consultato: adattamenti successivi guidati dagli esiti sono esplorativi.

Non ricaviamo recall o tassi di errore del paper dai suoi F1: non sono la stessa metrica.

In [ ]:
overall.to_csv(OUTPUT / 'model_metrics.csv', index=False)
per_class.to_csv(OUTPUT / 'class_metrics.csv', index=False)
paper.to_csv(OUTPUT / 'paper_table_iv.csv')
weighted_comparison.to_csv(OUTPUT / 'weighted_f1_vs_paper.csv', index=False)
class_comparison.to_csv(OUTPUT / 'class_f1_vs_paper.csv', index=False)
(OUTPUT / 'verification.json').write_text(json.dumps({
    'model_count': len(models), 'test_rows': len(expected_truth),
    'protocol_sha256': protocol_hash, 'same_targets_and_indices': True,
    'counts_and_metrics_recomputed': True, 'model_fits_in_comparison': 0,
    'paper_source': 'Divekar2018 Table IV NSL-KDD without SMOTE',
}, indent=2) + '\n', encoding='utf-8')
print('Confronto salvato: zero nuovi fit.')